## Imports

In [1]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

In [2]:
DATA_STORAGE_PATH: str = "../mock-cloud/storage/data"

In [3]:
DB_STORAGE_PATH: str = "../mock-cloud/db/hr_hub.db"

## Attrition Dataset

Data source: [Hr Analytics Job Prediction](https://www.kaggle.com/datasets/mfaisalqureshi/hr-analytics-and-job-prediction/data)

In [4]:
df: pd.DataFrame = pd.read_csv(f"{DATA_STORAGE_PATH}/raw/employee_attrition_raw.csv")
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   satisfaction_level     14999 non-null  float64
 1   last_evaluation        14999 non-null  float64
 2   number_project         14999 non-null  int64  
 3   average_montly_hours   14999 non-null  int64  
 4   time_spend_company     14999 non-null  int64  
 5   Work_accident          14999 non-null  int64  
 6   left                   14999 non-null  int64  
 7   promotion_last_5years  14999 non-null  int64  
 8   Department             14999 non-null  str    
 9   salary                 14999 non-null  str    
dtypes: float64(2), int64(6), str(2)
memory usage: 1.1 MB


In [5]:
df.head()

,satisfaction_level,last_evaluation,number_project,average_montly_hours,time_spend_company,Work_accident,left,promotion_last_5years,Department,salary
0,0.38,0.53,2,157,3,0,1,0,sales,low
1,0.80,0.86,5,262,6,0,1,0,sales,medium
2,0.11,0.88,7,272,4,0,1,0,sales,medium
3,0.72,0.87,5,223,5,0,1,0,sales,low
4,0.37,0.52,2,159,3,0,1,0,sales,low


In [6]:
# Column rename mapping
column_map: dict[str, str] = {
    "Department": "Department",
    "salary": "Salary",
    "number_project": "ActiveProjects",
    "average_montly_hours": "AvgMonthlyHours",
    "time_spend_company": "YearsAtCompany",
    "Work_accident": "WorkAccidents",
    "promotion_last_5years": "ReceivedPromotion",
    "last_evaluation": "LastEvaluation",
    "satisfaction_level": "SatisfactionScore",
    "left": "Attrition",
}

# Column order mapping
col_order: list[str] = [
    "Department",
    "Salary",
    "ActiveProjects",
    "AvgMonthlyHours",
    "YearsAtCompany",
    "WorkAccidents",
    "ReceivedPromotion",
    "LastEvaluation",
    "SatisfactionScore",
    "Attrition",
]

df_ordered: pd.DataFrame = df.rename(columns=column_map).reindex(columns=col_order)

In [7]:
df_ordered.head()

,Department,Salary,ActiveProjects,AvgMonthlyHours,YearsAtCompany,WorkAccidents,ReceivedPromotion,LastEvaluation,SatisfactionScore,Attrition
0,sales,low,2,157,3,0,0,0.53,0.38,1
1,sales,medium,5,262,6,0,0,0.86,0.80,1
2,sales,medium,7,272,4,0,0,0.88,0.11,1
3,sales,low,5,223,5,0,0,0.87,0.72,1
4,sales,low,2,159,3,0,0,0.52,0.37,1


In [8]:
df_ordered.ActiveProjects.max()

np.int64(7)

In [9]:
df_ordered.AvgMonthlyHours.max()

np.int64(310)

In [10]:
df_ordered.YearsAtCompany.max()

np.int64(10)

In [11]:
df_ordered.WorkAccidents.unique().tolist()

[0, 1]

In [12]:
df_ordered.ReceivedPromotion.unique().tolist()

[0, 1]

In [13]:
df_ordered.Attrition.unique().tolist()

[1, 0]

In [14]:
df_clean_types: pd.DataFrame = df_ordered.astype({
    "ActiveProjects": np.uint8,
    "AvgMonthlyHours": np.uint16,
    "WorkAccidents": bool,
    "ReceivedPromotion": bool,
    "LastEvaluation": np.float16,
    "SatisfactionScore": np.float16,
    "Attrition": bool
})

In [15]:
# Save ~55% memory usage by using appropriate data types
df_clean_types.info()

<class 'pandas.DataFrame'>
RangeIndex: 14999 entries, 0 to 14998
Data columns (total 10 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   Department         14999 non-null  str    
 1   Salary             14999 non-null  str    
 2   ActiveProjects     14999 non-null  uint8  
 3   AvgMonthlyHours    14999 non-null  uint16 
 4   YearsAtCompany     14999 non-null  int64  
 5   WorkAccidents      14999 non-null  bool   
 6   ReceivedPromotion  14999 non-null  bool   
 7   LastEvaluation     14999 non-null  float16
 8   SatisfactionScore  14999 non-null  float16
 9   Attrition          14999 non-null  bool   
dtypes: bool(3), float16(2), int64(1), str(2), uint16(1), uint8(1)
memory usage: 498.1 KB


In [16]:
df_clean_types.Department = df_clean_types.Department.replace({
    "technical": "engineering",
    "product_mng": "product_management",
    "RandD": "r&d"
})

In [17]:
df_clean_types.Department.unique().tolist()

['sales',
 'accounting',
 'hr',
 'engineering',
 'support',
 'management',
 'IT',
 'product_management',
 'marketing',
 'r&d']

In [18]:
df_clean_types.Department.value_counts()

Department
sales                 4140
engineering           2720
support               2229
IT                    1227
product_management     902
marketing              858
r&d                    787
accounting             767
hr                     739
management             630
Name: count, dtype: int64

In [19]:
df_clean_types.to_csv(f"{DATA_STORAGE_PATH}/processed/employee_attrition_processed.csv", index=False)

## DB creation and seeding

We need to create more synthetic data. We will use external LLM for generation of missing categorical fields, and statistical methods for numeric data imputation. This database will be the source of truth for our backend. All generate fields will have an illustrative purpose only, the original format will be used for training and prediction.

### Create random employees for DB

In [20]:
!uv run python3 generate_employees.py

  Total rows          : 14,999
  Unique emails       : 14,999
  CEO email           : matthew.diaz.ceo@company.com
  CEO manager_email   : nan

  Gender distribution:
Gender
M    0.6
F    0.4

  Department count:
Department
sales                 4140
engineering           2720
support               2229
IT                    1227
product_management     902
marketing              858
r&d                    787
accounting             767
hr                     739
management             630

  All managers → CEO  : True
Created employees.csv  ✅


In [21]:
df_employees: pd.DataFrame = pd.read_csv(f"{DATA_STORAGE_PATH}/processed/employees.csv")
df_employees.head()

,ID,FirstName,LastName,Gender,Email,Department,ManagerEmail,Laptop,Monitor,Headset,CreatedDate,UpdatedDate
0,e04bbdaa-52f9-48d3-8877-6caffa2657db,Matthew,Diaz,M,matthew.diaz.ceo@company.com,management,NaN,Dell Latitude 7440,True,True,2022-12-13,2023-10-01
1,7749efa5-fa2a-4b8f-81d5-a0e821086f21,Carol,Bennett,F,carol.bennett@company.com,management,matthew.diaz.ceo@company.com,"MacBook Pro 14""",True,False,2021-09-14,2021-10-30
2,128b6b02-a255-4816-9841-fd09ee894acb,Ryan,Clark,M,ryan.clark@company.com,management,matthew.diaz.ceo@company.com,Dell XPS 15,True,True,2018-11-03,2019-10-07
3,78c6ae1c-a7e6-471f-b301-5ac6d09b2624,Carolyn,Thompson,F,carolyn.thompson@company.com,management,matthew.diaz.ceo@company.com,"MacBook Pro 16""",True,True,2021-07-17,2021-08-27
4,f5649404-1d99-4c65-ba37-544edb568684,Kenneth,Adams,M,kenneth.adams@company.com,management,matthew.diaz.ceo@company.com,ThinkPad X1 Carbon,True,False,2020-02-17,2020-09-07


In [22]:
df_employees.Email.nunique()

14999

### Randomly assign employees to training data, match by department.

In [23]:
df_db: pd.DataFrame = pd.read_csv(f"{DATA_STORAGE_PATH}/processed/employee_attrition_processed.csv")
df_db.head()

,Department,Salary,ActiveProjects,AvgMonthlyHours,YearsAtCompany,WorkAccidents,ReceivedPromotion,LastEvaluation,SatisfactionScore,Attrition
0,sales,low,2,157,3,False,False,0.53,0.38,True
1,sales,medium,5,262,6,False,False,0.86,0.80,True
2,sales,medium,7,272,4,False,False,0.88,0.11,True
3,sales,low,5,223,5,False,False,0.87,0.72,True
4,sales,low,2,159,3,False,False,0.52,0.37,True


In [24]:
# Build a lookup: department → list of employee IDs
dept_to_ids = df_employees.groupby("Department")["ID"].apply(list).to_dict()

# For each row in df_db, sample a random ID from the matching department
rng = np.random.default_rng(96)

df_db.insert(0, "EmployeeID", df_db["Department"].apply(
    lambda dept: rng.choice(dept_to_ids[dept])
))

In [25]:
df_db["AttritionRisk"] = np.nan

In [26]:
df_db.head(1)

,EmployeeID,Department,Salary,ActiveProjects,AvgMonthlyHours,YearsAtCompany,WorkAccidents,ReceivedPromotion,LastEvaluation,SatisfactionScore,Attrition,AttritionRisk
0,5ca3df4e-21e6-411f-bc01-17b123fa0c7e,sales,low,2,157,3,False,False,0.53,0.38,True,NaN


### Seed database

In [27]:
engine = create_engine(f"sqlite:///{DB_STORAGE_PATH}", echo=False)

df_employees.drop(columns=["Department"])

with engine.begin() as conn:
    df_employees.to_sql("employees", conn, if_exists="replace", index=False)
    df_db.to_sql("employees_info", conn, if_exists="replace", index=False)

print("Done — tables written: employees, employees_info")

Done — tables written: employees, employees_info


In [28]:
with engine.connect() as conn:
    print(pd.read_sql("SELECT COUNT(*) as n FROM employees", conn))
    print(pd.read_sql("SELECT COUNT(*) as n FROM employees_info", conn))

       n
0  14999
       n
0  14999
